# Оценка ТЗ: трудоёмкость и приоритет уточнений

Методология: `metodology.pdf`. Для каждого компонента ChatGPT оценивает распределение
`[Zero, Small, Medium, Large]`; при отсутствии содержательных данных Python берёт prior меню.
`d = k × [0,1,3,10]`, `E[D] = Σpᵢdᵢ`, `U = √Σpᵢ(dᵢ−E[D])²`,
`Raw Priority = U`, `Priority = U / √c`. Сортировка по Priority — по убыванию.

Вероятности являются экспертными оценками. Потенциально снимаемая неопределённость равна U
при допущении, что ответ полностью определит уровень компонента. Это не гарантированная
экономия труда; стандартные отклонения компонентов нельзя просто складывать.


In [ ]:
%pip install "openai>=1.40,<3" "python-dotenv>=1,<2" "pandas>=2.1,<3" "pydantic>=2.8,<3" "python-docx>=1.1,<2" "pypdf>=5,<7" "tqdm>=4.66,<5"

In [1]:
from pathlib import Path
import sys

# Запуск из корня репозитория, evaluator или evaluator/evaluator_v2.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'evaluator' / 'evaluator_v2' / 'probability_evaluator.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри репозитория case-finder.')
sys.path.insert(0, str(ROOT))
BASE = ROOT / 'evaluator' / 'evaluator_v2'
from evaluator.evaluator_v2.probability_evaluator import (
    load_menu, load_spec, evaluate, summarize, save_results, top_questions,
    EFFORT, UNCERTAINTY, QUESTION,
)
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI
import os

# Ближайший .env имеет приоритет; значения ключей не выводятся.
for env_path in (BASE / '.env', BASE.parent / '.env', ROOT / '.env'):
    load_dotenv(env_path, override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Добавьте OPENAI_API_KEY в .env в корне репозитория или evaluator_v2.')
MODEL = os.getenv('OPENAI_LLM_MODEL', 'gpt-4.1-mini')
client = OpenAI(timeout=120, max_retries=2)


In [6]:
TZ_DIR = ROOT / 'evaluator' / 'TZ'
# Укажите файл ТЗ из evaluator/TZ; поддерживаются TXT, MD, DOCX и текстовый PDF.
TZ_PATH = TZ_DIR / '1 200 000 94900061_Приложение-№-1-к-извещению-ТЗ-чат-бот-Пассажирсервис.docx'  # Замените на имя вашего ТЗ.
MENU_PATH = BASE / 'chatbots_menu.csv'
OUTPUT_DIR = BASE / 'results'
BATCH_SIZE = 6
MAX_SPEC_CHARS = 180_000  # Текст никогда не обрезается автоматически.
TOP_N = 15

print('Доступные ТЗ:')
for path in sorted(TZ_DIR.rglob('*')):
    if path.suffix.lower() in {'.txt', '.md', '.docx', '.pdf'}:
        print(path.relative_to(TZ_DIR))


Доступные ТЗ:
1 200 000 94900061_Приложение-№-1-к-извещению-ТЗ-чат-бот-Пассажирсервис.docx
ТЗ - Сервис для создания виртуальных ассистентов (Electra 2.0).docx
ТЗ на второй этап Открытие.docx
ТЗ_на разработку и внедрение автоматизированного программного комплекса ЧАТ-БОТ ТСС.DOCX
Чат-бот Техническое задание.docx
Чатбот БФКО ТЗ v1_1.docx


In [7]:
menu = load_menu(MENU_PATH)
text = load_spec(TZ_PATH, TZ_DIR)
print(f'Загружено {len(text):,} символов ТЗ; компонентов меню: {len(menu)}')
with tqdm(total=len(menu), desc='Анализ ТЗ', unit='комп.') as bar:
    def progress(done, total):
        bar.update(done - bar.n)
    results = evaluate(client, MODEL, menu, text, batch_size=BATCH_SIZE,
                       max_spec_chars=MAX_SPEC_CHARS, progress=progress)
csv_path = save_results(results, OUTPUT_DIR, TZ_PATH.stem)
print(f'CSV сохранён: {csv_path}')
display(results)


Загружено 28,593 символов ТЗ; компонентов меню: 42


Анализ ТЗ:   0%|          | 0/42 [00:00<?, ?комп./s]

(0.0, 1.0, 0.0, 0.0)
explicit
1
False
----
(0.0, 0.0, 1.0, 0.0)
explicit
1
False
----
(0.0, 1.0, 0.0, 0.0)
explicit
1
False
----
(0.7, 0.18, 0.09, 0.03)
explicit
0
True
----
(0.75, 0.15, 0.07, 0.03)
explicit
0
True
----
(0.8, 0.1, 0.07, 0.03)
explicit
0
True
----
Исправление ответа ChatGPT: попытка 2/3
(0.0, 1.0, 0.0, 0.0)
explicit
1
False
----
(0.0, 0.0, 1.0, 0.0)
explicit
1
False
----
(0.75, 0.15, 0.08, 0.02)
none
0
True
----
(1.0, 0.0, 0.0, 0.0)
explicit
1
False
----
(1.0, 0.0, 0.0, 0.0)
explicit
1
False
----
(1.0, 0.0, 0.0, 0.0)
explicit
1
False
----
(0.0, 1.0, 0.0, 0.0)
explicit
1
False
----
(0.0, 1.0, 0.0, 0.0)
explicit
1
False
----
(0.0, 0.95, 0.05, 0.0)
explicit
0
True
----
(0.0, 0.5, 0.5, 0.0)
explicit
0
True
----
(0.0, 1.0, 0.0, 0.0)
explicit
1
False
----
(0.0, 0.0, 1.0, 0.0)
explicit
1
False
----
Исправление ответа ChatGPT: попытка 2/3
(0.0, 1.0, 0.0, 0.0)
explicit
1
False
----
(0.05, 0.8, 0.15, 0.0)
partial
0
True
----
(0.0, 0.0, 1.0, 0.0)
explicit
1
False
----
(0.0, 0.0, 1

,Блок,Компонент,Распределение оценки сложности Zero/Small/Medium/Large,Ожидаемые затраты (человекодни),Неопределенность (человекодни),Raw Priority,Priority,Уточняющие вопросы,Коментарий
0,Интеграции и действия,Agentic tool use / выбор инструментов,"[0.5, 0.3, 0.15, 0.05]",6.25,11.277744,11.277744,7.974569,Должен ли ассистент сам выбирать и последовате...,Основание: «Сервис должен бесшовно интегрирова...
1,Интеграции и действия,Выполнение действий от имени пользователя,"[0.25, 0.5, 0.2, 0.05]",6.40,8.708616,8.708616,6.157922,Должен ли ассистент только читать данные или т...,Основание: «Сервис должен иметь возможность пе...
2,Качество и аналитика,Автоматическая оценка качества (Evals),"[0.25, 0.5, 0.2, 0.05]",6.40,8.708616,8.708616,6.157922,Требуется ли формальное и регулярное измерение...,Основание: «- Требования к отчетности и тестир...
3,Enterprise и эксплуатация,Админ-панель,"[0.35, 0.5, 0.1, 0.05]",5.20,8.681014,8.681014,6.138404,Что бизнес-пользователи должны уметь менять бе...,Основание: «Сервис должен иметь возможность ги...
4,Инфраструктура,Развертывание,"[0.02, 0.55, 0.32, 0.11]",10.44,11.040217,11.040217,5.520109,"Где должна работать система: обычное облако, к...",Основание: «Работы выполняются удаленно. Испол...
5,Знания и RAG,Цитирование и доказательность ответов,"[0.1, 0.45, 0.35, 0.1]",5.00,5.422177,5.422177,5.422177,Насколько строго ответ должен быть связан с ис...,Основание: «В интерфейсе платформы Чат-бота пр...
6,Инфраструктура,Наблюдаемость / мониторинг,"[0.1, 0.55, 0.27, 0.08]",6.48,7.538541,7.538541,5.330553,Нужно ли видеть только ошибки и latency или тр...,Основание: «Сервис должен иметь встроенную сис...
7,Безопасность,PII / конфиденциальные данные,"[0.0, 0.6, 0.3, 0.1]",10.00,10.620734,10.620734,5.310367,"Какие персональные, коммерчески чувствительные...",Основание: «Требования к Заказчику при использ...
8,Enterprise и эксплуатация,Конфигурируемость без разработки,"[0.65, 0.25, 0.08, 0.02]",4.14,9.475252,9.475252,4.737626,Нужно ли дать бизнесу no-code/low-code возможн...,Основание: «Сервис должен иметь возможность ги...
9,Enterprise и эксплуатация,Мультитенантность,"[0.75, 0.15, 0.07, 0.03]",3.30,9.143850,9.143850,4.571925,Нужно ли обслуживать несколько независимых орг...,"Основание: «Исполнитель – организация, с котор..."


In [8]:
print(f'Ожидаемые суммарные затраты: {results[EFFORT].sum():.2f} человекодней')
print(f'Компонентов, требующих уточнения: {results[QUESTION].str.strip().ne("").sum()}')
print('Потенциально снимаемая неопределённость указана для каждого вопроса отдельно.')
display(top_questions(results, TOP_N).style.format({
    'Priority': '{:.3f}',
    'Потенциально снимаемая неопределенность (человекодни)': '{:.2f}',
}))


Ожидаемые суммарные затраты: 194.47 человекодней
Компонентов, требующих уточнения: 21
Потенциально снимаемая неопределённость указана для каждого вопроса отдельно.


,Блок,Компонент,Уточняющие вопросы,Priority,Потенциально снимаемая неопределенность (человекодни)
0,Интеграции и действия,Agentic tool use / выбор инструментов,"Должен ли ассистент сам выбирать и последовательно вызывать несколько инструментов для достижения цели, или набор действий заранее фиксирован и прост?",7.975,11.28
1,Интеграции и действия,Выполнение действий от имени пользователя,"Должен ли ассистент только читать данные или также изменять их/совершать бизнес-операции, и насколько критичны эти действия и нужны ли подтверждения/approvals/rollback?",6.158,8.71
2,Качество и аналитика,Автоматическая оценка качества (Evals),"Требуется ли формальное и регулярное измерение качества ассистента на эталонных наборах с интеграцией в CI/CD и блокировкой релизов, или достаточно ручного тестирования и базовой отчетности?",6.158,8.71
3,Enterprise и эксплуатация,Админ-панель,"Что бизнес-пользователи должны уметь менять без разработчиков: только несколько настроек или также знания, промпты, пользователей, роли, сценарии и релизы?",6.138,8.68
4,Инфраструктура,Развертывание,"Где должна работать система: обычное облако, корпоративный Kubernetes/private cloud или полностью on-prem/air-gapped контур?",5.520,11.04
5,Знания и RAG,Цитирование и доказательность ответов,Насколько строго ответ должен быть связан с источниками: достаточно ссылки на документ или нужна проверяемая трассировка отдельных утверждений до конкретных фрагментов?,5.422,5.42
6,Инфраструктура,Наблюдаемость / мониторинг,Нужно ли видеть только ошибки и latency или требуется сквозная трассировка каждого LLM/RAG/tool шага с алертами и интеграцией в корпоративный monitoring/SIEM?,5.331,7.54
7,Безопасность,PII / конфиденциальные данные,"Какие персональные, коммерчески чувствительные или регулируемые данные будет обрабатывать система и какие требования есть к хранению, residency, masking и аудиту?",5.310,10.62
8,Enterprise и эксплуатация,Конфигурируемость без разработки,"Нужно ли дать бизнесу no-code/low-code возможность самостоятельно создавать или существенно перестраивать ассистентов, workflow, tools и политики?",4.738,9.48
9,Enterprise и эксплуатация,Мультитенантность,"Нужно ли обслуживать несколько независимых организаций/подразделений и насколько строго должны быть разделены их данные, настройки, ключи, модели и биллинг?",4.572,9.14


In [9]:
print('ChatGPT готовит общее заключение…')
conclusion = summarize(client, MODEL, text, results)
display(Markdown(conclusion))
report_path = OUTPUT_DIR / f'{TZ_PATH.stem}_заключение_v2.md'
report_path.write_text(conclusion, encoding='utf-8')
print(f'Заключение сохранено: {report_path}')


ChatGPT готовит общее заключение…


Техническое задание содержит развернутое описание системы интеллектуального чат-бота с интеграцией LLM, RAG, сценарной логикой, базой знаний и модулем операторской поддержки. Многие ключевые функциональные блоки раскрыты подробно: каналы коммуникации (Jivo, соцсети, виджеты), сценарии с переходами, аналитика, роли и права, интеграции с CRM, требования к SLA и безопасности. В целом, это проект со средней степенью сложности — не пирожок для одночеловеческой разработки, но и без элементов сверхсложной масштабируемости, мультиорганизационной мультиарендности или сложных ML-пайплайнов. Наиболее значимые пробелы связаны с уточнением характера выбора и последовательности действий ассистента (Agentic tool use), глубиной конфигурируемости администраторов (панель управления, no-code), детализацией инфраструктуры (облако, on-prem, Kubernetes), строгими требованиями к безопасности (PII, защита от инъекций), управлением версий и автоматическим тестированием (CI/CD и evals). Также неясен объем профилирования пользователей — от нескольких фиксированных параметров до сложной, долговременной персонализации. Интеграции и действия, мониторинг и оповещения, а также оптимизация стоимости LLM — обозначены, но требуют дополнительной детализации по уровню управления и автоматизации. Отсутствуют задачи мультиязычности, голосового ввода/вывода и мультимодальности. Общее впечатление: ТЗ покрывает основной минимальный и средний функционал интеллектуального чат-бота для бизнеса с LLM и RAG, но оставляет вопросы по масштабу, глубине адаптации и эксплуатационным процессам. Риски связаны с недостаточной проработкой некоторых ключевых аспектов реализации (интеграция действий ассистента, безопасность, CI/CD, инфраструктура), что требует уточняющих ответов для сокращения неопределенностей и риска срыва сроков и бюджета.

Заключение сохранено: /home/elwis/Projects/case-finder/evaluator/evaluator_v2/results/1 200 000 94900061_Приложение-№-1-к-извещению-ТЗ-чат-бот-Пассажирсервис_заключение_v2.md
